# Media Framing LLM Showcase

This notebook is designed for a lecturer-facing walkthrough of the final synchronous media-framing classification run.

It does four things:
- loads the final LLM classification output
- gives a compact overview of category patterns and source differences
- selects a small set of representative examples for each category
- adds English translations for presentation use while keeping the German original visible

The translation step is intentionally limited to the curated showcase sample and cached locally.


In [ ]:
from pathlib import Path
import hashlib
import html
import json
import re
import sys

import pandas as pd
import matplotlib.pyplot as plt
import requests

from IPython.display import HTML, Markdown, display


try:
    import jinja2  # noqa: F401
    HAS_JINJA2 = True
except ImportError:
    HAS_JINJA2 = False

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass

pd.set_option('display.max_colwidth', 220)

PROJECT_ROOT = Path('/Users/MattisHaumann/Dev/Thesis')
RESULTS_PATH = PROJECT_ROOT / '2a_NER/outputs/batch_media_framing/thesis_final/normal_api_run/media_framing_thesis_sync_results.csv'
SHOWCASE_DIR = PROJECT_ROOT / '2a_NER/outputs/batch_media_framing/thesis_final/showcase'
SHOWCASE_DIR.mkdir(parents=True, exist_ok=True)

TRANSLATION_CACHE_PATH = SHOWCASE_DIR / 'media_framing_lecturer_translation_cache.csv'
SHOWCASE_EXPORT_PATH = SHOWCASE_DIR / 'media_framing_lecturer_examples.csv'

EXAMPLES_PER_CATEGORY = 3
RUN_TRANSLATIONS = True
TRANSLATION_MODEL = 'gpt-5-mini'
TRANSLATION_TIMEOUT = 180

CATEGORY_ORDER = [
    'POSITIONS-/PARTEILICHKEITS-BIAS',
    'VERZERRUNG/MANIPULATION',
    'DISINFORMATION/FALSCHDARSTELLUNG',
    'VERSAGEN/INKOMPETENZ',
    'NEUTRAL',
    'IRRELEVANT',
]

CATEGORY_ENGLISH = {
    'POSITIONS-/PARTEILICHKEITS-BIAS': 'Positional / Partisan Bias',
    'VERZERRUNG/MANIPULATION': 'Distortion / Manipulation',
    'DISINFORMATION/FALSCHDARSTELLUNG': 'Disinformation / Falsehood',
    'VERSAGEN/INKOMPETENZ': 'Failure / Incompetence',
    'NEUTRAL': 'Neutral',
    'IRRELEVANT': 'Irrelevant',
}

CATEGORY_DESCRIPTIONS_EN = {
    'POSITIONS-/PARTEILICHKEITS-BIAS': 'The outlet is framed as politically aligned, loyal to an ideology, or serving particular interests.',
    'VERZERRUNG/MANIPULATION': 'The outlet is accused of selective framing, omission, contextual distortion, or manipulative presentation.',
    'DISINFORMATION/FALSCHDARSTELLUNG': 'The outlet is accused of lying, spreading fake news, or publishing false claims as facts.',
    'VERSAGEN/INKOMPETENZ': 'The outlet is criticized as shallow, professionally weak, or failing at basic journalistic work.',
    'NEUTRAL': 'The media actor is merely named, cited, or referenced without evaluative framing.',
    'IRRELEVANT': 'The regex hit does not actually function as a media reference in context.',
}

SOURCE_LABELS = {
    'RT_de': 'RT',
    'Tichys_Einblick': 'Tichys Einblick',
}

assert RESULTS_PATH.exists(), f'Results file not found: {RESULTS_PATH}'
print(f'Results path: {RESULTS_PATH}')
print(f'Showcase output dir: {SHOWCASE_DIR}')


In [ ]:
def normalize_source_label(value: object) -> str:
    if pd.isna(value):
        return ''
    value = str(value).strip()
    return SOURCE_LABELS.get(value, value)


def normalize_whitespace(text: object) -> str:
    if pd.isna(text):
        return ''
    return re.sub(r'\s+', ' ', str(text)).strip()


def build_context_excerpt(context: str, evidence: str, radius: int = 260, fallback: int = 420) -> str:
    context = normalize_whitespace(context)
    evidence = normalize_whitespace(evidence)
    if not context:
        return ''

    if evidence:
        match = re.search(re.escape(evidence), context, flags=re.IGNORECASE)
        if match:
            start = max(0, match.start() - radius)
            end = min(len(context), match.end() + radius)
            excerpt = context[start:end].strip()
            if start > 0:
                excerpt = '… ' + excerpt
            if end < len(context):
                excerpt = excerpt + ' …'
            return excerpt

    if len(context) <= fallback:
        return context
    return context[:fallback].rsplit(' ', 1)[0].strip() + ' …'


results_df = pd.read_csv(RESULTS_PATH)
results_df['source_label'] = results_df['source'].map(normalize_source_label)
results_df['title_de'] = results_df['Title'].map(normalize_whitespace)
results_df['hit_text'] = results_df['hit_text'].map(normalize_whitespace)
results_df['evidence_de'] = results_df['evidence'].map(normalize_whitespace)
results_df['context_window'] = results_df['context_window'].map(normalize_whitespace)
results_df['category'] = pd.Categorical(results_df['category'], categories=CATEGORY_ORDER, ordered=True)
results_df['category_en'] = results_df['category'].map(CATEGORY_ENGLISH)
results_df['evidence_present'] = results_df['evidence_de'].ne('')
results_df['context_excerpt_de'] = results_df.apply(
    lambda row: build_context_excerpt(row['context_window'], row['evidence_de']),
    axis=1,
)
results_df['title_key'] = results_df['title_de'].str.lower()
results_df['excerpt_len'] = results_df['context_excerpt_de'].str.len()
results_df['evidence_len'] = results_df['evidence_de'].str.len()
results_df['is_bias_label'] = ~results_df['category'].isin(['NEUTRAL', 'IRRELEVANT'])

overview = pd.Series(
    {
        'classified hits': len(results_df),
        'unique row_id values': results_df['row_id'].nunique(),
        'unique sources': results_df['source_label'].nunique(),
        'unique categories': results_df['category'].nunique(),
        'rows with explicit evidence spans': int(results_df['evidence_present'].sum()),
    }
)

display(overview.to_frame('value'))
display(
    results_df[
        ['source_label', 'title_de', 'hit_text', 'category', 'evidence_de', 'context_excerpt_de']
    ].head(5)
)


## Clean Overview

The two views below are the ones most useful in class:
- how often each category appears overall
- how category mixes differ by outlet


In [ ]:
category_summary = (
    results_df.groupby(['category', 'category_en'], observed=False)
    .agg(
        hits=('hit_id', 'size'),
        unique_articles=('row_id', 'nunique'),
        outlets=('source_label', 'nunique'),
        evidence_spans=('evidence_present', 'sum'),
    )
    .reset_index()
)
category_summary['share_of_hits'] = category_summary['hits'] / len(results_df)
category_summary['evidence_share_within_category'] = (
    category_summary['evidence_spans'] / category_summary['hits']
)

overview_table = category_summary[
    [
        'category_en',
        'category',
        'hits',
        'share_of_hits',
        'unique_articles',
        'outlets',
        'evidence_spans',
        'evidence_share_within_category',
    ]
].copy()

if HAS_JINJA2:
    display(
        overview_table.style.format(
            {
                'share_of_hits': '{:.1%}',
                'evidence_share_within_category': '{:.1%}',
            }
        )
    )
else:
    display(overview_table)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

plot_summary = category_summary.copy()
axes[0].bar(plot_summary['category_en'], plot_summary['hits'], color='#2c7fb8')
axes[0].set_title('Classification labels across all hits')
axes[0].set_ylabel('Number of hits')
axes[0].tick_params(axis='x', rotation=35)

bias_only = plot_summary[~plot_summary['category'].isin(['NEUTRAL', 'IRRELEVANT'])].copy()
axes[1].bar(bias_only['category_en'], bias_only['hits'], color='#d95f0e')
axes[1].set_title('Bias accusation labels only')
axes[1].set_ylabel('Number of hits')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

source_category_share = pd.crosstab(
    results_df['source_label'],
    results_df['category_en'],
    normalize='index',
).reindex(columns=[CATEGORY_ENGLISH[c] for c in CATEGORY_ORDER])

if HAS_JINJA2:
    display(
        source_category_share.style.format('{:.1%}').background_gradient(cmap='Blues')
        .set_caption('Within-outlet category mix')
    )
else:
    display(source_category_share)


## Category Glossary

This compact English glossary is useful when explaining the annotation scheme to a lecturer who has not seen the German prompt taxonomy.


In [ ]:
codebook_df = pd.DataFrame(
    [
        {
            'Category (DE)': category,
            'Category (EN)': CATEGORY_ENGLISH[category],
            'How to explain it in class': CATEGORY_DESCRIPTIONS_EN[category],
        }
        for category in CATEGORY_ORDER
    ]
)

display(codebook_df)


## Curated Showcase Sample

The goal here is not to inspect all rows, but to pick representative examples that are easy to explain aloud.

Selection logic:
- prefer rows with clear evidence spans for accusation labels
- keep excerpts short enough for presentation
- avoid repeating the same article when possible
- try to keep some source diversity inside each category


In [ ]:
def example_score(row: pd.Series) -> float:
    context_score = max(0.0, 1 - abs(row['excerpt_len'] - 380) / 380)
    title_bonus = 0.15 if row['title_de'] else 0.0
    mention_bonus = 0.15 if row['hit_text'] else 0.0

    if row['category'] in ['NEUTRAL', 'IRRELEVANT']:
        return context_score + title_bonus + mention_bonus

    evidence_score = 2.0 if row['evidence_present'] else -3.0
    evidence_score += max(0.0, 1 - abs(row['evidence_len'] - 45) / 45)
    return context_score + evidence_score + title_bonus + mention_bonus


results_df['showcase_score'] = results_df.apply(example_score, axis=1)


def select_examples_for_category(group: pd.DataFrame, n: int) -> pd.DataFrame:
    ranked = group.sort_values(
        ['showcase_score', 'evidence_len', 'excerpt_len'],
        ascending=False,
    ).copy()

    selected_indices = []
    used_titles = set()
    used_sources = set()
    source_diversity_available = ranked['source_label'].nunique() >= n

    for _, row in ranked.iterrows():
        if len(selected_indices) >= n:
            break
        if row['title_key'] in used_titles:
            continue
        if source_diversity_available and row['source_label'] in used_sources:
            continue

        selected_indices.append(row.name)
        used_titles.add(row['title_key'])
        used_sources.add(row['source_label'])

    if len(selected_indices) < n:
        for _, row in ranked.iterrows():
            if len(selected_indices) >= n:
                break
            if row.name in selected_indices:
                continue
            if row['title_key'] in used_titles:
                continue

            selected_indices.append(row.name)
            used_titles.add(row['title_key'])

    return ranked.loc[selected_indices].head(n)


showcase_examples = pd.concat(
    [
        select_examples_for_category(
            results_df[results_df['category'] == category].copy(),
            EXAMPLES_PER_CATEGORY,
        )
        for category in CATEGORY_ORDER
    ],
    ignore_index=True,
)

showcase_examples['example_rank'] = showcase_examples.groupby('category', observed=False).cumcount() + 1
showcase_examples['translation_key'] = showcase_examples.apply(
    lambda row: hashlib.md5(
        '||'.join(
            [
                row['category'],
                row['title_de'],
                row['evidence_de'],
                row['context_excerpt_de'],
            ]
        ).encode('utf-8')
    ).hexdigest(),
    axis=1,
)

display(
    showcase_examples[
        [
            'category_en',
            'example_rank',
            'source_label',
            'title_de',
            'hit_text',
            'evidence_de',
            'showcase_score',
        ]
    ]
)

showcase_examples.to_csv(SHOWCASE_EXPORT_PATH, index=False, encoding='utf-8')
print(f'Showcase sample saved to: {SHOWCASE_EXPORT_PATH}')


## English Translations For The Lecturer Version

This cell translates only the curated examples, not the full dataset. The result is cached to CSV, so reruns should usually be free after the first successful pass.


In [ ]:
HELPER_DIR = PROJECT_ROOT / '2a_NER'
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from media_framing_batch_utils import extract_output_text_from_responses_body, read_env_value

TRANSLATION_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'title_en': {'type': 'string'},
        'evidence_en': {'type': 'string'},
        'excerpt_en': {'type': 'string'},
    },
    'required': ['title_en', 'evidence_en', 'excerpt_en'],
}

TRANSLATION_INSTRUCTIONS = (
    'Translate the provided German media-framing example into fluent English for an academic presentation. '
    'Preserve the accusation strength, framing, and tone. '
    'Do not summarize or soften the wording. '
    'Translate the evidence as a short, telling English phrase. '
    'If the evidence field is empty, return an empty string for evidence_en. '
    'Return valid JSON only.'
)


def load_translation_cache(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame(
        columns=['translation_key', 'title_en', 'evidence_en', 'excerpt_en']
    )


def save_translation_cache(cache_df: pd.DataFrame, path: Path) -> None:
    if cache_df.empty:
        return
    cache_df = cache_df.drop_duplicates(subset=['translation_key'], keep='last').copy()
    cache_df.to_csv(path, index=False, encoding='utf-8')


def request_translation(session: requests.Session, row: pd.Series) -> dict:
    payload = {
        'model': TRANSLATION_MODEL,
        'instructions': TRANSLATION_INSTRUCTIONS,
        'input': (
            f"Title (German):\n{row['title_de']}\n\n"
            f"Evidence span (German):\n{row['evidence_de']}\n\n"
            f"Excerpt (German):\n{row['context_excerpt_de']}"
        ),
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'showcase_translation',
                'strict': True,
                'schema': TRANSLATION_SCHEMA,
            }
        },
    }

    response = session.post(
        'https://api.openai.com/v1/responses',
        json=payload,
        timeout=TRANSLATION_TIMEOUT,
    )
    response.raise_for_status()
    response_json = response.json()
    return json.loads(extract_output_text_from_responses_body(response_json))


translation_cache = load_translation_cache(TRANSLATION_CACHE_PATH)
cache_lookup = (
    translation_cache.drop_duplicates(subset=['translation_key'], keep='last')
    .set_index('translation_key')
    .to_dict('index')
    if not translation_cache.empty
    else {}
)

api_key, api_key_source = read_env_value('OPENAI_API_KEY', project_root=PROJECT_ROOT)
print(f'Cached translations already available: {len(cache_lookup):,}')
print(f'OPENAI_API_KEY source: {api_key_source}')

session = None
if RUN_TRANSLATIONS and api_key:
    session = requests.Session()
    session.headers.update(
        {
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        }
    )
elif RUN_TRANSLATIONS:
    print('OPENAI_API_KEY not found. Translation step will use cache only.')
else:
    print('RUN_TRANSLATIONS = False. Translation step will use cache only.')

translated_rows = []
translation_errors = []

for _, row in showcase_examples.iterrows():
    key = row['translation_key']

    if key in cache_lookup:
        translated_rows.append({'translation_key': key, **cache_lookup[key]})
        continue

    if session is None:
        translated_rows.append(
            {
                'translation_key': key,
                'title_en': '',
                'evidence_en': '',
                'excerpt_en': '',
            }
        )
        continue

    try:
        translated = request_translation(session, row)
        record = {
            'translation_key': key,
            'title_en': translated.get('title_en', '').strip(),
            'evidence_en': translated.get('evidence_en', '').strip(),
            'excerpt_en': translated.get('excerpt_en', '').strip(),
        }
        translated_rows.append(record)
        cache_lookup[key] = record
        translation_cache = pd.concat([translation_cache, pd.DataFrame([record])], ignore_index=True)
        save_translation_cache(translation_cache, TRANSLATION_CACHE_PATH)
    except Exception as exc:
        translation_errors.append(
            {
                'translation_key': key,
                'category': row['category'],
                'source_label': row['source_label'],
                'title_de': row['title_de'],
                'error': str(exc),
            }
        )
        translated_rows.append(
            {
                'translation_key': key,
                'title_en': '',
                'evidence_en': '',
                'excerpt_en': '',
            }
        )

translations_df = pd.DataFrame(translated_rows).drop_duplicates('translation_key', keep='last')
showcase_examples = showcase_examples.drop(
    columns=['title_en', 'evidence_en', 'excerpt_en'],
    errors='ignore',
).merge(translations_df, on='translation_key', how='left')

for column in ['title_en', 'evidence_en', 'excerpt_en']:
    showcase_examples[column] = showcase_examples[column].fillna('').astype(str)

showcase_examples.to_csv(SHOWCASE_EXPORT_PATH, index=False, encoding='utf-8')
print(f'Updated showcase export: {SHOWCASE_EXPORT_PATH}')
print(f'Translation cache: {TRANSLATION_CACHE_PATH}')

if translation_errors:
    display(pd.DataFrame(translation_errors))

display(
    showcase_examples[
        ['category_en', 'example_rank', 'source_label', 'title_en', 'evidence_en']
    ]
)


## Final Lecturer-Facing Render

Each card keeps the German original visible, but surfaces the English translation directly underneath it.

For the accusation labels, the German evidence span is highlighted in context. For `Neutral` and `Irrelevant`, no evidence span is expected.


In [ ]:
CATEGORY_COLORS = {
    'POSITIONS-/PARTEILICHKEITS-BIAS': '#b2182b',
    'VERZERRUNG/MANIPULATION': '#ef8a62',
    'DISINFORMATION/FALSCHDARSTELLUNG': '#542788',
    'VERSAGEN/INKOMPETENZ': '#2166ac',
    'NEUTRAL': '#4d4d4d',
    'IRRELEVANT': '#7f7f7f',
}


def safe_text(value: object, placeholder: str = '—') -> str:
    value = '' if pd.isna(value) else str(value).strip()
    return value if value else placeholder


def highlight_evidence_html(text: str, evidence: str) -> str:
    text = normalize_whitespace(text)
    evidence = normalize_whitespace(evidence)
    if not text:
        return '—'
    if not evidence:
        return html.escape(text)

    match = re.search(re.escape(evidence), text, flags=re.IGNORECASE)
    if not match:
        return html.escape(text)

    return (
        html.escape(text[: match.start()])
        + '<mark>'
        + html.escape(text[match.start() : match.end()])
        + '</mark>'
        + html.escape(text[match.end() :])
    )


DISPLAY_CSS = '''
<style>
.showcase-grid {
    display: grid;
    grid-template-columns: 1fr;
    gap: 16px;
    margin: 12px 0 28px 0;
}
.showcase-card {
    border: 1px solid #d9d9d9;
    border-left: 8px solid var(--accent-color);
    border-radius: 10px;
    padding: 16px 18px;
    background: #ffffff;
    box-shadow: 0 1px 4px rgba(0, 0, 0, 0.04);
}
.showcase-meta {
    font-size: 12px;
    color: #555;
    margin-bottom: 10px;
    letter-spacing: 0.02em;
    text-transform: uppercase;
}
.showcase-title {
    font-size: 17px;
    font-weight: 700;
    margin-bottom: 10px;
}
.showcase-pair {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
    margin-top: 10px;
}
.showcase-box {
    background: #fafafa;
    border: 1px solid #ececec;
    border-radius: 8px;
    padding: 12px;
}
.showcase-label {
    font-size: 12px;
    font-weight: 700;
    color: #444;
    margin-bottom: 6px;
    text-transform: uppercase;
    letter-spacing: 0.03em;
}
.showcase-text {
    font-size: 14px;
    line-height: 1.55;
    color: #111;
}
.showcase-evidence {
    font-weight: 700;
    color: #111;
}
mark {
    background: #fff2a8;
    padding: 0 2px;
}
@media (max-width: 900px) {
    .showcase-pair {
        grid-template-columns: 1fr;
    }
}
</style>
'''

display(HTML(DISPLAY_CSS))


def render_example_card(row: pd.Series) -> str:
    accent = CATEGORY_COLORS.get(row['category'], '#999999')
    evidence_de = safe_text(
        row['evidence_de'],
        placeholder='No explicit evidence span (expected for Neutral / Irrelevant).',
    )
    evidence_en = safe_text(
        row['evidence_en'],
        placeholder='No explicit evidence span.' if row['category'] in ['NEUTRAL', 'IRRELEVANT'] else '[translation pending]',
    )
    title_en = safe_text(row['title_en'], placeholder='[translation pending]')
    excerpt_en = safe_text(row['excerpt_en'], placeholder='[translation pending]')
    mention = safe_text(row['hit_text'])
    excerpt_de_html = highlight_evidence_html(row['context_excerpt_de'], row['evidence_de'])

    return f'''
    <div class="showcase-card" style="--accent-color: {accent};">
        <div class="showcase-meta">Example {int(row['example_rank'])} | {html.escape(row['source_label'])} | matched media mention: {html.escape(mention)}</div>
        <div class="showcase-title">{html.escape(row['category_en'])}</div>
        <div class="showcase-pair">
            <div class="showcase-box">
                <div class="showcase-label">German title</div>
                <div class="showcase-text">{html.escape(safe_text(row['title_de']))}</div>
            </div>
            <div class="showcase-box">
                <div class="showcase-label">English title</div>
                <div class="showcase-text">{html.escape(title_en)}</div>
            </div>
        </div>
        <div class="showcase-pair">
            <div class="showcase-box">
                <div class="showcase-label">Evidence span in German</div>
                <div class="showcase-text showcase-evidence">{html.escape(evidence_de)}</div>
            </div>
            <div class="showcase-box">
                <div class="showcase-label">Evidence span in English</div>
                <div class="showcase-text showcase-evidence">{html.escape(evidence_en)}</div>
            </div>
        </div>
        <div class="showcase-pair">
            <div class="showcase-box">
                <div class="showcase-label">German excerpt</div>
                <div class="showcase-text">{excerpt_de_html}</div>
            </div>
            <div class="showcase-box">
                <div class="showcase-label">English excerpt</div>
                <div class="showcase-text">{html.escape(excerpt_en)}</div>
            </div>
        </div>
    </div>
    '''


for category in CATEGORY_ORDER:
    category_examples = showcase_examples[showcase_examples['category'] == category].copy()
    if category_examples.empty:
        continue

    display(Markdown(f"## {CATEGORY_ENGLISH[category]}"))
    display(
        Markdown(
            f"**German category label:** `{category}`  \n"
            f"{CATEGORY_DESCRIPTIONS_EN[category]}"
        )
    )
    cards_html = ''.join(render_example_card(row) for _, row in category_examples.iterrows())
    display(HTML(f'<div class="showcase-grid">{cards_html}</div>'))

print(f'Curated lecturer showcase exported to: {SHOWCASE_EXPORT_PATH}')
